One more recommendation

I would actually rename the notebook.

Instead of:

feature_removal_audit.ipynb

I'd use:

feature_removal_validation.ipynb

or

feature_removal_justification.ipynb

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path
import yaml

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

: 

In [ ]:
DATA = Path("../dataset/processed/extracted_ecg_features.csv")

REMOVED = Path("../outputs/metadata/removed_features.csv")

CONFIG = Path("../config.yaml")

df = pd.read_csv(DATA)
removed = pd.read_csv(REMOVED)

with open(CONFIG) as f:
    config = yaml.safe_load(f)

print(df.shape)
removed.head()

## Dataset Overview

Before investigating the removals, let's understand how many features were removed and how they are distributed.

In [ ]:
removed["reason"].value_counts()

In [ ]:
removed["reason"].value_counts().plot.bar(figsize=(8,4))
plt.title("Reasons for Feature Removal")
plt.show()

The majority of removed features belong to only two categories.

Let's investigate the count features first.

In [ ]:
count_cols = [c for c in df.columns if c.endswith("_count")]

len(count_cols), count_cols[:10]

In [ ]:
df[count_cols].head()

Do these features actually contain different information?

In [ ]:
corr = df[count_cols].corr()

plt.figure(figsize=(12,10))
sns.heatmap(corr,cmap="viridis")
plt.title("Correlation Between Count Features")
plt.show()

In [ ]:
corr.unstack().sort_values(ascending=False).head(30)

If almost every count feature is perfectly correlated, they are likely measuring the same quantity and retaining all of them is unnecessary.

In [ ]:
reference = df["QRS_Dur_Global_count"]

comparison = []

for col in count_cols:
    if col == "QRS_Dur_Global_count":
        continue

    comparison.append({
        "feature":col,
        "equal_rows":(reference==df[col]).mean()*100,
        "correlation":reference.corr(df[col])
    })

comparison = pd.DataFrame(comparison)
comparison.sort_values("correlation",ascending=False)

In [ ]:
comparison.describe()

### Interpretation

Based on the correlation and exact-match analysis, the count features appear highly redundant.

Keeping a single representative feature is therefore reasonable.

# ST_Elev_*_iqr Features

Unlike the count features, these were removed because previous validation reported them as unreliable.

Let's inspect whether the current dataset itself provides evidence for that decision.

In [ ]:
st_iqr = [c for c in df.columns if c.startswith("ST_Elev") and c.endswith("_iqr")]

st_iqr

In [ ]:
df[st_iqr].describe().T

In [ ]:
df[st_iqr].isna().mean().sort_values(ascending=False)

In [ ]:
df[st_iqr].var().sort_values()

In [ ]:
plt.figure(figsize=(15,6))
sns.boxplot(data=df[st_iqr])
plt.xticks(rotation=90)
plt.show()

### Observation

This notebook can inspect the statistical behaviour of the ST elevation IQR features.

However, it cannot independently prove that these features should be removed.

The original justification came from the earlier validation phase.

Therefore the evidence shown here should be considered supportive rather than definitive.

In [ ]:
summary = pd.DataFrame({
    "Category":["Count Features","ST_Elev_IQR"],
    "Notebook Evidence":[
        "Strong",
        "Limited"
    ],
    "Reason":[
        "High redundancy",
        "Prior validation"
    ],
    "Recommendation":[
        "Remove",
        "Remove (based on prior validation)"
    ]
})

summary

# Final Conclusion

Count features:
- Strong evidence of redundancy.
- Removal is well supported by this notebook.

ST_Elev_*_iqr features:
- The notebook shows descriptive statistics only.
- The primary justification comes from the previous validation study.
- Those earlier results should be cited when documenting this preprocessing decision.